# VS Code + Colab：Qwen LoRA
在 VS Code 打开此文件，右上角选择 Colab GPU 内核。先运行环境检查，再逐格执行。无需手动上传 ZIP。
本 Notebook 内嵌公开训练数据与脚本；运行解包格会将其发送到 Colab。模型在云端下载。尚未验证实际远程连接。


In [ ]:
import platform, sys, subprocess, shutil
import torch
print('OS:', platform.system(), 'Python:', sys.executable)
print('CUDA:', torch.cuda.is_available())
print('PyTorch:', torch.__version__, 'CUDA build:', torch.version.cuda)
if not torch.cuda.is_available():
    smi = shutil.which('nvidia-smi')
    result = subprocess.run([smi, '-L'], capture_output=True, text=True) if smi else None
    if result and result.returncode == 0 and 'GPU' in result.stdout:
        print(result.stdout)
        raise RuntimeError('已检测到 NVIDIA GPU，但 PyTorch 无法使用。请保留以上版本信息，检查 CUDA 环境；不要反复新建服务器。')
    raise RuntimeError('当前运行时未检测到可用 NVIDIA GPU。这不代表连接失败。在内核选择器中选择 Colab → New Colab Server，并选择 GPU（可用时选 T4），不要继续选择原 CPU 服务器。')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


In [ ]:
from pathlib import Path
import base64, io, zipfile, hashlib, json, os
payload = '...'
raw = base64.b64decode(payload)
assert hashlib.sha256(raw).hexdigest() == 'e4e0ace7c2b48fb28705f1045989f206686f4c7c1b1fb4d8754fb5d1e9a6e4a2'
root = Path('/content/intent_router')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for entry in z.infolist():
        assert (root / entry.filename).resolve().is_relative_to(root.resolve())
    z.extractall(root)
os.chdir(root)
for name, sha in json.loads(Path('bundle_manifest.json').read_text()).items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == sha
print('已解包并验证训练数据与脚本：', root)


安装固定框架版本，保留 Colab 自带的 CUDA torch/vision/audio。若依赖解析失败，不要强行忽略依赖。安装在子进程中，后续训练也使用新进程。

In [ ]:
import subprocess, sys, importlib.metadata as metadata
constraints = []
for name in ['torch', 'torchvision', 'torchaudio']:
    try: constraints.append(name + '==' + metadata.version(name))
    except metadata.PackageNotFoundError: pass
Path('colab_constraints.txt').write_text('\n'.join(constraints))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-c', 'colab_constraints.txt', 'llamafactory==0.9.4'], check=True)
print('框架安装完成')


In [ ]:
# 下载与本机相同 revision 的模型；不需要付费 API。
subprocess.run([sys.executable, 'download_model.py'], check=True)
subprocess.run([sys.executable, '-c', "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name()); print('BF16:',torch.cuda.is_bf16_supported())"], check=True)


挂载 Drive 保存 checkpoint，避免运行时回收后全部丢失。授权只在 Colab 页面进行。每次运行新目录，checkpoint 每 25 步保存一次，保留最近两个；最后一次未保存的进度仍可能丢失。

In [ ]:
from google.colab import drive
from datetime import datetime
drive.mount('/content/drive')
result_root = Path('/content/drive/MyDrive/CIE6032-intent-router')
result_root.mkdir(exist_ok=True)


先运行这一格：10 个 optimizer step 的 GPU 短测。完成后查看 gpu_measurement.json 的显存和时间；短测 adapter 不用于正式效果对比。

In [ ]:
smoke_out = result_root / ('smoke-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
subprocess.run([sys.executable, 'colab_train.py', '--smoke', '--output', str(smoke_out)], check=True)
print((smoke_out / 'gpu_measurement.json').read_text())


短测正常后，手动运行下一格开始 1000 条正式训练。5000 条是另一轮：稍后将 SIZE 改成 5000，再运行此格。两组使用不同输出目录。当前评估是 teacher-forced dev loss；任务准确率需之后用统一生成评测，不能用 loss 替代。

In [ ]:
SIZE = 1000
train_out = result_root / (f'train-{SIZE}-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
subprocess.run([sys.executable, 'colab_train.py', '--size', str(SIZE), '--output', str(train_out)], check=True)
print((train_out / 'gpu_measurement.json').read_text())


如断线：重新连接、上传、安装、下载模型、挂载 Drive，跳过上述新训练格。把恢复格中的两个路径填成原输出目录和其最新完整 checkpoint 目录；参数 SIZE 必须与原运行一致。

In [ ]:
# 恢复示例：取消注释并填写真实路径后执行
# old_out = result_root / 'train-1000-实际时间'
# checkpoint = old_out / 'checkpoint-100'
# subprocess.run([sys.executable, 'colab_train.py', '--size', '1000', '--output', str(old_out), '--resume', str(checkpoint)], check=True)


训练结束后把最近一次输出打包保存在 Drive，用 Drive 网页下载或扩展的远程文件浏览器取回。这里不用网页专用 files.download()。


In [ ]:
import shutil
last = Path(Path('colab_last_output.txt').read_text())
archive = shutil.make_archive(str(result_root / last.name), 'zip', last)
print('结果保存在 Google Drive：', archive)
